In [ ]:
from collections.abc import Mapping
import pandas as pd
from pymarc.record import Record, Field, Subfield, Indicators
import re

In [ ]:
class LazyDict(Mapping):
    def __init__(self, *args, **kw):
        self._raw_dict = dict(*args, **kw)

    def __getitem__(self, key):
        func, kwargs = self._raw_dict.__getitem__(key)
        return func(**kwargs)

    def __iter__(self):
        return iter(self._raw_dict)

    def __len__(self):
        return len(self._raw_dict)

In [ ]:
dtype = {
    "Date 1": str,
    "Date 2": str,
    "NACO identifier for name": str,
    "Date of publication in Arabic or Roman numerals": str,
    "NACO identifier for additional name (#1)": str,
    "NACO identifier for additional name (#2)": str,
    "NACO identifier for additional name (#3)": str,
    "NACO identifier for additional name (#4)": str,
    "NACO identifier for additional name (#5)": str,
    "NACO identifier for additional name (#6)": str,
    "NACO identifier for additional name (#7)": str
}
marc_df = pd.read_csv("../data/processed/non_gt_outputs/batch_260130/260130_trial_records-ATG.csv", encoding="utf8", nrows=12, dtype=dtype)

In [ ]:
def parse_subject_col(s: str) -> Field:
    if not s:
        raise ValueError("Trying to parse empty column")
        
    split = s.split()
    field = s[0]
    ind2 = s[1]
    sf_split = split[-1].split("$")[1:]
    subfields = [Subfield(sf[0], sf[1:]) for sf in sf_split]
    return Field(tag=field, indicators=Indicators("#",ind2), subfields=subfields)

In [ ]:
def parse_name(tag: str, type_of_name: str, name: str, rel_to_res: str, naco_id: str, rel_res: str="") -> Field:
    
    digits = re.compile(r"\d")
    name_i1 = {"Personal name - forename first": "0", "Personal name - surname first": "1"}.get(type_of_name, "#")
    name_has_date = digits.search(name.split(",")[-1])
    if name_has_date:
        name = ",".join(name.split(",")[:-1])
        name_date = name.split(",")[-1]
    else:
        name_date = ""
        
    return Field(tag=tag, indicators=Indicators(name_i1,"#"), subfields=[Subfield("a", name), Subfield("d", name_date), Subfield("e", rel_to_res), Subfield("t", rel_res), Subfield("1", naco_id)])

In [ ]:
def row_to_marc(row):
    # only create fields with data
    # remove any blank subfields at the end
    # fixed_length_data_elements
    row = row.dropna()
    
    flde_00_05 = "      "
    flde_06 = row.get("Type of publication date").split()[0]
    flde_07_10 = row.get("Date 1")
    flde_11_14 = row.get("Date 2", "    ")
    flde_15_17 = row.get("Country of publication") + " "
    flde_18_34 = "    ||    |||| ||"
    flde_35_37 = "may"

    field_lookup = LazyDict({
        "Aleph system number": (Field, dict(tag="001", data=row.get("Aleph system number"))),
        "Type of publication date": (Field, dict(tag="008", data=flde_00_05+flde_06+flde_07_10+flde_11_14+flde_15_17+flde_18_34+flde_35_37)),
        "Main language": (Field, dict(tag="041", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Main language"))] + [Subfield("a", lang) for lang in row.get("Additional languages", "").split(";")])),
        "Name": (parse_name, dict(tag="100", type_of_name=row.get("Type of name"), name=row.get("Name"), rel_to_res=row.get("Relationship to resource"), naco_id=row.get("NACO identifier for name"))),
        "Title": (Field, dict(tag="245", indicators=Indicators("0",row.get("\xa0")), subfields=[Subfield("a", row.get("Title")), Subfield("b", row.get("Subtitle", "") + " " + row.get("Parallel title", "")), Subfield("c", row.get("Statement of responsibility"))])),
        "Variant title": (Field, dict(tag="246", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Variant title"))])),
        "Edition": (Field, dict(tag="250", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Edition"))])),
        "Place of publication": (Field, dict(tag="264", indicators=Indicators("#","1"), subfields=[Subfield("a", row.get("Place of publication")), Subfield("b", row.get("Publisher")), Subfield("c", row.get("Date of publication in Arabic or Roman numerals"))])),
        "Place of manufacture": (Field, dict(tag="264", indicators=Indicators("#","3"), subfields=[Subfield("a", row.get("Place of manufacture")), Subfield("b", row.get("Manufacturer")), Subfield("c", row.get("Date of manufacture in Arabic or Roman numerals"))])),
        "Extent": (Field, dict(tag="300", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Extent")), Subfield("b", row.get("Illustrations")), Subfield("c", row.get("Dimensions"))])),
        "Main content type": (Field, dict(tag="336", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Main content type"))])),
        "Carrier type": (Field, dict(tag="338", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Carrier type"))])),
        "Series title": (Field, dict(tag="490", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Series title")), Subfield("v", row.get("Number within series"))])),
        "General notes": (Field, dict(tag="500", indicators=Indicators("#","#"), subfields=[Subfield("a", general_notes)])),
        "Bibliography etc. note": (Field, dict(tag="504", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Bibliography etc. note"))])),
        "Citation/references note": (Field, dict(tag="510", indicators=Indicators("3","#"), subfields=[Subfield("a", row.get("Citation/references note"))])),
        "Language note": (Field, dict(tag="546", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Language note"))])),
        "Ownership and custodial history": (Field, dict(tag="561", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Ownership and custodial history"))])),
        "Identifying markings": (Field, dict(tag="562", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Identifying markings"))])),
        "Binding Information": (Field, dict(tag="563", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Binding Information"))])),
        "Subject or genre": (parse_subject_col, dict(s=row.get("Subject or genre"))),
        "Unnamed: 50": (parse_subject_col, dict(s=row.get("Unnamed: 50"))),
        "Unnamed: 51": (parse_subject_col, dict(s=row.get("Unnamed: 51"))),
        "Unnamed: 52": (parse_subject_col, dict(s=row.get("Unnamed: 52"))),
        "Unnamed: 53": (parse_subject_col, dict(s=row.get("Unnamed: 53"))),
        "Unnamed: 54": (parse_subject_col, dict(s=row.get("Unnamed: 54"))),
        "Additional name (#1)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#1)"), name=row.get("Additional name (#1).1"), rel_to_res=row.get("Relationship to resource (#1)"), rel_res="Related resource (#1)", naco_id=row.get("NACO identifier for additional name (#1)"))),
        "Additional name (#2)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#2)"), name=row.get("Additional name (#2).1"), rel_to_res=row.get("Relationship to resource (#2)"), rel_res="Related resource (#2)", naco_id=row.get("NACO identifier for additional name (#2)"))),
        "Additional name (#3)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#3)"), name=row.get("Additional name (#3).1"), rel_to_res=row.get("Relationship to resource (#3)"), rel_res="Related resource (#3)", naco_id=row.get("NACO identifier for additional name (#3)"))),
        "Additional name (#4)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#4)"), name=row.get("Additional name (#4).1"), rel_to_res=row.get("Relationship to resource (#4)"), rel_res="Related resource (#4)", naco_id=row.get("NACO identifier for additional name (#4)"))),
        "Additional name (#5)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#5)"), name=row.get("Additional name (#5).1"), rel_to_res=row.get("Relationship to resource (#5)"), rel_res="Related resource (#5)", naco_id=row.get("NACO identifier for additional name (#5)"))),
        "Additional name (#6)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#6)"), name=row.get("Additional name (#6).1"), rel_to_res=row.get("Relationship to resource (#6)"), rel_res="Related resource (#6)", naco_id=row.get("NACO identifier for additional name (#6)"))),
        "Additional name (#7)": (parse_name, dict(tag="700", type_of_name=row.get("Additional name (#7)"), name=row.get("Additional name (#7).1"), rel_to_res=row.get("Relationship to resource (#7)"), rel_res="Related resource (#7)", naco_id=row.get("NACO identifier for additional name (#7)"))),
        # "Method of acquisition": (Field, dict(tag="", indicators=Indicators("#","#"), subfields=[Subfield("", row.get("Method of acquisition"))])),
        "Shelfmark": (Field, dict(tag="852", indicators=Indicators(4,1), subfields=[Subfield("j", row.get("Shelfmark")), Subfield("q", row.get("Damage"))])),
        "ID": (Field, dict(tag="985", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("ID"))])),
        "Cataloguer's note": (Field, dict(tag="985", indicators=Indicators("#","#"), subfields=[Subfield("a", row.get("Cataloguer's note"))]))        
    })

    fields = [field_lookup.get(col, None) for col in row.index]
    strawberry_fields = []
    for f in fields:
        if f is None:
            continue
        elif f.data:
            strawberry_fields.append(f)
        else:
            for sf in f.subfields:
                if sf.value is None or sf.value == "":
                    f.delete_subfield(sf.code)
            if f.subfields:
                strawberry_fields.append(f)

    ldr = "      am a2200229 a 4500"
    record = Record(leader=ldr, force_utf8=True)
    [record.add_ordered_field(f) for f in strawberry_fields]
    record.add_ordered_field(Field(tag="040", indicators=Indicators("#","#"), subfields=[Subfield("a", "Uk"), Subfield("b", "end"), Subfield("c", "Uk"), Subfield("e", "rda"), Subfield("e", "dcrmr")]))
    record.add_ordered_field(Field(tag="042", indicators=Indicators("#","#"), subfields=[Subfield("a", "ukblproject")]))
    record.add_ordered_field(Field(tag="337", indicators=Indicators("#","#"), subfields=[Subfield("a", "unmediated"), Subfield("2", "rdamedia")]))
    return record

Columns and inclusions to add:

HL added
LDR (6) = a   (Language material)  
LDR (7) = m (Monograph/item)  
LDR (17) = #  
LDR (18) = a  

HL added
040 (Cataloguing source) = \\$aUk\\$beng\\$cUk\\$erda\\$edcrmr

HL added
042 (Authentication code) = \\$aukblproject

300 \\$c (see below)

340 \\$m (see below)

Current BL policy is that early print has bibliographic format (foliation) assigned if applicable (see reference to 300 \\$c and 340 \\$m here):

​xlsx icon​​ Cataloguing Hub.xlsx​​

https://www.loc.gov/marc/bibliographic/bd300.html
https://www.loc.gov/marc/bibliographic/bd340.html

HL added
337 (Media type) = \\$aunmediated\\$2rdamedia

561 (Ownership and custodial history re record three on the sample sheet) = From the Bibliothèque Favre, 1888, item 353(50)\\$d5Uk  or Given by Bibliothèque Favre, 1888, item 353(50)\\$5Uk (561 in the bib. is relevant for monos only. For serial publications the 561 would be added to the relevant holdings record)

This is broadly in line with guidance on recording custodial history for early print set out here:

https://bsc.rbms.info/DCRMR/additional-notes/Custodial-history-of-item/

https://www.loc.gov/marc/bibliographic/bd561.html

655 #7 \\$aLithographic papers\\$2rbmscv    - IF "Lithographed" represents the form of resource being described in the 500 field (see below). If not, then there is no requirement for the aforementioned 655  

https://rbms.info/vocabularies/paper/tr225.htm

HL - 913 we could set as a single date for a given batch, if we have a column for that I can convert, or Annabel can just tell me what the date is?
913 (date the record was finished i.e. if 17th of February 2026 ) = \\$\\$a Y \\$\\$d 20260217    No indicators required

Columns to change: 

HL added
100 Indicator 1 and Indicator 2 missing

HL added
245 Indicator 1 missing

HL added
264 Indicator 1 and 2 (Publication ) missing

HL added
264 Indicator 1 and 2 (Manufacture) missing

HL fixed in post-processing
500 \\$a General notes - first letter should be made upper case i.e. Lithographed (unless 655 is not more appropriate (see above).

HL added - fix the comma setting in post-processing rather than MARC-ifying
510 \\$a First indicator value required:

https://www.loc.gov/marc/bibliographic/bd510.html

Examples include citation, date of citation, title of resource described and date of pub of resource. But title and date of pub are elsewhere in the record : e.g. Barzanji Makna Bugis, 1896 so maybe the y are not necessary as part of the 510.  

If they are necessary as part of the 510, add a comma after citation and the title and before the date i.e. Proudfoot, 1993: Barzanji Makna Bugis, 1896. 

 If a location within the source can be recorded, then this would be consistent with current early print policy and it would be recorded in 510 \\$c (see 510 link above).

Also, if Proudfoot is part of a serial think about recording 1993 in 510 \\$b.  

6XX 

Current BL policy is that early print collections have LCSH assigned rather than FAST as subject work and RBMS terms for genre  (see page 2 guidance and last example on page 4 here) :

​Subject.pdf​

Please note LCSH Indicator 1 would be # and Indicator 2 would be 0 

HL Added
700 (Indicator 1 and Indicator 2)

Column with a query:

1XX and 7XX \\$1- NACO identifier for name (\\$1 = Real World Object URI )    -  Uniform Resource Identifier required? https://www.loc.gov/marc/bibliographic/ecbdcntf.html

The bibliographic application profile does not currently list 1XX or 7XX $1 for any resource types:

​xlsx icon Cataloguing Hub.xlsx


In [ ]:
records = marc_df.iloc[2:].apply(row_to_marc, axis=1).to_list()

In [ ]:
records[0].as_marc()

In [ ]:
marc_df.dropna(axis=1, how="all", subset=[x for x in range(2, 12)])